In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

print("--- NLP ENVIRONMENT READY ---")

--- NLP ENVIRONMENT READY ---


In [2]:
movies = pd.read_csv('/Users/dhruvitjalodhara/programming/ML Practice/Movie Recommendation System/movie dataset/processed/filtered_movies.csv')

print(f"Dataset Loaded Successfully!")
print(f"Total Movies to Vectorize: {movies.shape[0]}")
print(f"Available Columns: {list(movies.columns)}")

Dataset Loaded Successfully!
Total Movies to Vectorize: 16116
Available Columns: ['movieId', 'title', 'genres', 'tags', 'metadata_soup']


In [3]:
# Sanity check: Count missing values in your text target
print(f"Missing values in metadata_soup: {movies['metadata_soup'].isnull().sum()}")

# Fill any hidden NaN cells with an empty string defensively
movies['metadata_soup'] = movies['metadata_soup'].fillna('')

# Inspect a sample soup entry to verify configuration
print("\n--- SAMPLE METADATA SOUP ENTRY ---")
print(movies['metadata_soup'].iloc[0])

Missing values in metadata_soup: 0

--- SAMPLE METADATA SOUP ENTRY ---
Adventure Animation Children Comedy Fantasy animation friendship toys animation Disney Pixar toys CGI classic disney pixar lots_of_heart Tom_Hanks clever funny Pixar witty animation comedy kids pixar Tom_Hanks Toy_Story toys animated fun jealousy funny witty adventure animation comedy family fantasy John_Lasseter USA family_film friendship toys Disney dolls National_Film_Registry buddy_movie Tom_Hanks witty pixar adventure clever enemies_become_friends family Family_cartoon family_relationships feel-good first_of_series friends friendship fun fun_family_movie HEROIC_MISSION humorous jealousy kids kids_movie loyal_friend loyalty redemption reflection rescue_mission rivalry selflessness teamwork unlikely_friendships unusual_friendship witty Os_dois_viram adventure animated animation cgi comedy Disney family fantasy friendship imdb_top_250 Pixar Tom_Hanks witty classic Disney friendship pixar accepting_reality emotiona

In [4]:
# 1. Initialize the vectorizer ignoring common English filler words
tfidf = TfidfVectorizer(stop_words='english')

# 2. Fit and transform the metadata soup columns
tfidf_matrix = tfidf.fit_transform(movies['metadata_soup'])

print("--- TEXT VECTORIZATION COMPLETE ---")
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")

--- TEXT VECTORIZATION COMPLETE ---
TF-IDF Matrix Shape: (16116, 124302)


In [5]:
from sklearn.metrics.pairwise import linear_kernel

print("Computing text similarity matrix...")
# linear_kernel calculates the raw dot product, which equals cosine similarity for normalized TF-IDF vectors
content_sim_matrix = linear_kernel(tfidf_matrix, tfidf_matrix)

print("--- CONTENT SIMILARITY MATRIX MATRIX BUILT ---")
print(f"Matrix Shape: {content_sim_matrix.shape}")

Computing text similarity matrix...
--- CONTENT SIMILARITY MATRIX MATRIX BUILT ---
Matrix Shape: (16116, 16116)


In [6]:
# 1. Create your quick lookup maps using the movies dataframe index
movie_to_idx = pd.Series(movies.index, index=movies['title'])
idx_to_movie = {v: k for k, v in movie_to_idx.items()}

search_title = "Kung Fu Panda (2008)"

if search_title in movie_to_idx.index:
    target_idx = movie_to_idx[search_title]
    
    # Extract text similarity scores for this specific movie
    sim_scores = content_sim_matrix[target_idx]
    
    # Sort positions from highest to lowest similarity
    sorted_indices = np.argsort(sim_scores)[::-1]
    
    # Grab the top 5 closest text matches (skipping index 0)
    top_5_matches = sorted_indices[1:6]
    
    print(f"NLP Content-Based Recommendations for '{search_title}':")
    print("-" * 60)
    
    for rank, idx in enumerate(top_5_matches, 1):
        recommended_title = idx_to_movie[idx]
        score = sim_scores[idx]
        print(f"{rank}. {recommended_title} (Text Match Score: {score:.2%})")
else:
    print(f"Could not locate '{search_title}'. Check title configuration strings.")

NLP Content-Based Recommendations for 'Kung Fu Panda (2008)':
------------------------------------------------------------
1. Kung Fu Panda 3 (2016) (Text Match Score: 52.30%)
2. Kung Fu Panda 2 (2011) (Text Match Score: 44.43%)
3. Ice Age (2002) (Text Match Score: 42.77%)
4. Drunken Master (Jui kuen) (1978) (Text Match Score: 42.66%)
5. Kung Fu Jungle (2014) (Text Match Score: 41.70%)
